# 2. Pré-processamento dos Dados — Thermal Predictor

Agora que a planilha já foi reestruturada e os dados já foram explorados, é possível iniciar o pré-processamento dos dados. Nessa seção haverá a o tratamento de missing values, identificação de outliers e a correção de colunas e linhas com erro.

### Para que serve o Pré-processamento dos Dados?

O pré-processamento dos dados serve para organizar, limpar e transformar os dados brutos, corrigindo inconsistências, tratando valores ausentes e removendo informações inadequadas. Ele impacta diretamente a fase de preparação dos dados, pois garante que as informações estejam padronizadas e em um formato adequado para a análise e aplicação dos modelos, contribuindo para resultados mais confiáveis. 

### Alterações diretas na planilha

Antes de iniciar todo o processo de limpeza dos dados, a planilha dos dados foi alterada diretamente no excel, devido a erros de digitação e a mal formatação da planilha.
Principais alterações:
- Correção do número do componente, em que havia um erro no comando no excel, na qual a ordem dos números contia um erro, resultando em #REF!;

- Conversão do 'Monitor', presente na coluna de 'active_limit' para 1000. Essa alteração teve como finalidade manter um padrão para apresentar apenas números nessa coluna, evitando o uso de strings. O número 1000 foi escolhido, pois como o 'Monitor' significa que não há um limite de temperatura, foi escolhido um número que seria tecnicamente impossível de ser alcançada nesses testes;

- Na planilha havia número em formato de porcentagem, o que prejudicaria a compreensão da tabela, portanto foi alterada para o formato númerico padrão;

- As linhas 92 a 93 das Runs 1 até 18 foram coladas no fim da planilha, caso não retiradas, poderiam comprometer com a visualização do DataFrame.

## 1. Carregar planilha já reestruturada

In [ ]:
import pandas as pd
df = pd.read_csv('../data/dataset_long.csv')

# 2. Tratamento de dados nulos

Na reestruturação da planilha, os valores nulos de temperatura já foram retirados. Essa foi considerada pelo grupo a melhor estratégia, pois o tipo de teste pode alterar significativamente as temperaturas do fogão. Dessa forma, inserir valores como média, mediana ou moda poderia impactar significativamente a análise dos dados, principalmente na fase de preparação dos dados.

Entretanto, ainda existem alguns dados nulos que precisam ser tratados, como os valores referentes ao tipo de teste. Nesses casos, será inserido o valor **"Não informado"**, mantendo os registros na base sem atribuir uma informação que não foi registrada.

In [10]:
# Definir as colunas categóricas que serão alteradas

colunas_categoricas = [
    'reason',
    'blower_direction',
    'housing',
    'board_position',
    'run_case',
    'losses',
    'deflector_hole',
    'material_coil_support'
]

# Preencher valores nulos nas colunas categóricas com 'Não informado'

df[colunas_categoricas] = df[colunas_categoricas].fillna('Não informado')

# Detectar linhas com valores 'Não informado' nas colunas categóricas
linhas_com_nao_informado = df[
    df[colunas_categoricas].eq('Não informado').any(axis=1)
]
# Exibir o número de linhas com valores 'Não informado' e as próprias linhas
print(f"Linhas com 'Não informado': {len(linhas_com_nao_informado)}")
linhas_com_nao_informado

Linhas com 'Não informado': 552


,run_id,tc_number,tc_name,key_area,active_limit,temp,reason,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support
0,Run 1,1,"Door Lock Motor Ambient 1/2"" Below",Mainback,155,99.35,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
1,Run 1,2,Door Lock Microswitch Body,Mainback,136,114.72,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
2,Run 1,3,Convect Motor Winding,Mainback,165,89.93,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
3,Run 1,7,Storage Drawer (Metal Above),Mainback,150,167.87,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
4,Run 1,8,"L2, Wire, Power Cord",Mainback,60,53.65,Não informado,To back,Open,Center,Pyro,No Losses,Open,Aluminium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1690,Run 23,74,Left Encoder,CT Knobs,105,43.19,New Baseline - Hole Tray,To Front,Open - Updated,Center,Broil C2,Não informado,Closed,Aluminium
1691,Run 23,75,Left Center Encoder,CT Knobs,105,57.36,New Baseline - Hole Tray,To Front,Open - Updated,Center,Broil C2,Não informado,Closed,Aluminium
1692,Run 23,76,Ambient Between Right Encoders,CT Knobs,105,47.65,New Baseline - Hole Tray,To Front,Open - Updated,Center,Broil C2,Não informado,Closed,Aluminium
1693,Run 23,77,Right Center Encoder,CT Knobs,105,40.66,New Baseline - Hole Tray,To Front,Open - Updated,Center,Broil C2,Não informado,Closed,Aluminium


## 3. Tratamento de Outliers

Não é possível remover os outliers deste conjunto de dados, pois essa ação poderia comprometer a aplicação do machine learning. Cada tipo de teste apresenta comportamentos diferentes para cada peça e, por isso, é esperado que existam padrões distintos entre os testes. Essa variação natural resulta em valores considerados outliers quando os dados são analisados em conjunto, mas que representam corretamente as condições específicas de cada teste.

Um exemplo são os testes com `run_case` **Pyro**, que apresentam temperaturas muito baixas nas Coil temperatures, próximas de 50 °C. Em contraste, os testes com `run_case` **Broil C2** apresentam temperaturas muito altas, que podem chegar a aproximadamente 220 °C. Essas diferenças não indicam necessariamente erros ou anomalias nos dados, mas sim comportamentos próprios de cada tipo de teste.

Portanto, os outliers não devem ser removidos, pois são informações relevantes para que o modelo de machine learning aprenda as diferentes condições e comportamentos observados nos testes.

In [24]:
# Calcular o primeiro e terceiro quartil (Q1 e Q3) da temperatura para cada tipo de termopar
q1_por_tc = df.groupby('tc_name')['temp'].transform('quantile', 0.25)
q3_por_tc = df.groupby('tc_name')['temp'].transform('quantile', 0.75)

# Calcular o intervalo interquartil (IQR)
iqr_por_tc = q3_por_tc - q1_por_tc

# Definir o limite inferior e superior para identificar possíveis outliers
limite_inferior = q1_por_tc - 1.5 * iqr_por_tc
limite_superior = q3_por_tc + 1.5 * iqr_por_tc

# Indentidica como outlier os valores abaixo do limite inferior ou acima do limite superior
outliers = (df['temp'] < limite_inferior) | (df['temp'] > limite_superior)

# Filtrar os outliers, agrupá-los por tc_name e contar o total em cada grupo
total_outliers_por_tc = (
    df.loc[outliers]
    .groupby('tc_name')
    .size()
    .reset_index(name='total_outliers')
    .sort_values('total_outliers', ascending=False)
)

# Exibir o total de outliers por tipo de termopar
total_outliers_por_tc

,tc_name,total_outliers
47,IPC Vertical Duct Inlet (Bottom),15
5,AC Blower Housing,13
19,Coil FR,12
51,Power On Indicator Body,11
62,Top Duct Inlet,10
...,...,...
20,Convect Motor Winding,1
46,IPC Housing Surface,1
42,IPC Blower (Air Outlet Temp),1
38,IGBT 6,1


## 4. Tratamento de dados confusos ou insuficientes

Durante a análise, foram identificados dados que não puderam ser interpretados com segurança, pois a equipe não possui informações suficientes para confirmar seu significado. Nesses casos, é importante registrar a inconsistência e evitar a criação de valores ou interpretações que possam comprometer a análise. 

### 4.1 Active Limit do Turbo Duct / AC Blower Outlet

O `active_limit` desse componente está registrado como **"Monitor (150 °C)"**. Essa informação é confusa, pois não está claro se o componente deve apenas ser monitorado, sem possuir uma temperatura limite definida, ou se o seu limite de temperatura é de 150 °C. Como não há informações suficientes para decidir entre essas interpretações, o dado deve ser mantido conforme registrado na planilha original. 
O valor irá ser trocado por 150, por enquanto, a fim de manter o padrão das temperaturas em números, porém poderá ser trocado futuramente.

### 4.2 Componentes sem informações

Também foram encontrados componentes que ocupam uma linha inteira em branco ou apresentam pouquíssimas informações. Um exemplo é o **Dedicated IPC/CKTP Outlet**, que está completamente em branco. Como esse componente não possui dados suficientes para contribuir com a análise, ele foi retirado por completo durante a reestruturação dos dados.

### 4.3 Oven RTD Connector Body

O componente **Oven RTD Connector Body** possui um `active_limit` de 150 °C, porém todas as temperaturas registradas ultrapassam esse limite e ficam em torno de 200 °C. Portanto, os dados indicam uma possível inconsistência entre o limite informado e as temperaturas observadas.

Além disso, existe uma observação associada ao componente:

> **ID#AAACE8Xo3xk (2026-08-07 14:52:06)**  
> *CAD for RTD assembly (W11633068) passes through the backframe; likely will not see 150 C temps so remove for early stage simulation.*

Essa observação sugere que o componente provavelmente não atingirá 150 °C e recomenda sua remoção em uma simulação inicial. No entanto, a orientação ainda é subjetiva para o tratamento dos dados deste projeto, pois não está claro se o componente deve ser removido da análise, se o `active_limit` deve ser alterado ou se os valores devem ser mantidos para representar as medições reais.

Diante dessa incerteza, o grupo optou por não remover nem alterar esses dados neste momento. O componente e sua inconsistência devem ser mantidos registrados e encaminhados para validação com a equipe responsável antes de qualquer decisão definitiva.

In [ ]:
# Replace robusto: qualquer variação de 'Monitor...' vira 150. O texto original tem
# espaçamento inconsistente entre as linhas (ex.: 'Monitor (150 C)' vs 'Monitor( 150 C)'),
# por isso a substituição usa regex em vez de comparar a string exata.
df['active_limit'] = df['active_limit'].replace(r'Monitor.*', 150, regex=True)

# Filtrar as linhas do componente Turbo Duct / AC Blower Outlet
turbo_duct_ac_blower_outlet = df[
    df['tc_name'].astype('string').str.contains(
        'Turbo Duct / AC Blower Outlet',
        case=False,
        na=False,
    )
]

# Exibir somente as 5 primeiras e as 5 últimas linhas encontradas
amostra_turbo_duct_ac_blower_outlet = pd.concat([
    turbo_duct_ac_blower_outlet.head(3),
    turbo_duct_ac_blower_outlet.tail(3),
]).drop_duplicates()

# Exibir o total de linhas encontradas
print(f"Total de linhas encontradas: {len(turbo_duct_ac_blower_outlet)}")
amostra_turbo_duct_ac_blower_outlet

## 5. Salvar Alterações

## 6. Codificação de variáveis categóricas e normalização de variáveis quantitativas

Esta seção implementa a atividade WAD 4.2.2, com o rigor apontado em revisão: valores de dados corrigidos antes de qualquer transformação, outliers verificados com evidência (não afirmados), separação explícita entre features e a variável-alvo, divisão treino/teste para evitar vazamento de dados, e um dataset final limpo e validado.

A variável-alvo do projeto é `temp` (Seção 4.1.3: "o modelo deverá prever a temperatura com margem de erro máxima de 10%"). As demais colunas foram separadas em: identificadores/texto livre excluídos (`run_id`, `tc_number`, `tc_name`, `reason`, `losses`), categóricas preditoras (`key_area`, `blower_direction`, `housing`, `board_position`, `run_case`, `material_coil_support`, `deflector_hole`) e quantitativa preditora (`active_limit`).

In [ ]:
TARGET = 'temp'
ID_COLS = ['run_id', 'tc_number', 'tc_name', 'reason', 'losses']
CATEGORICAS = [
    'key_area', 'blower_direction', 'housing',
    'board_position', 'run_case',
    'material_coil_support', 'deflector_hole',
]
NUMERICAS = ['active_limit']

# Garante que nenhuma coluna do dataset foi esquecida na classificação acima --
# se uma coluna nova for adicionada ao CSV no futuro, este assert falha ruidosamente
# em vez de o pipeline ignorá-la silenciosamente.
colunas_classificadas = set(ID_COLS) | set(CATEGORICAS) | set(NUMERICAS) | {TARGET}
colunas_faltando = set(df.columns) - colunas_classificadas
colunas_extras = colunas_classificadas - set(df.columns)
assert not colunas_faltando, f"Colunas do dataset não classificadas: {colunas_faltando}"
assert not colunas_extras, f"Colunas classificadas que não existem no dataset: {colunas_extras}"

print(f"Alvo: {TARGET}")
print(f"Identificadores/texto livre excluídos: {ID_COLS}")
print(f"Categóricas a codificar: {CATEGORICAS}")
print(f"Numéricas (features) a normalizar: {NUMERICAS}")

### 6.1 Conversão de `active_limit` para tipo numérico

A Seção 4 já substituiu qualquer valor `"Monitor..."` por `150`, mas a coluna continua com tipo texto (mistura de strings numéricas com o inteiro já convertido). Convertendo explicitamente com `pd.to_numeric(errors='raise')`: se sobrar algum valor não numérico não identificado antes, o erro aponta exatamente qual, em vez de deixá-lo silenciosamente virar `NaN`.

In [ ]:
df['active_limit'] = pd.to_numeric(df['active_limit'], errors='raise').astype(float)
df['active_limit'].describe()

### 6.2 Correção de inconsistência em `material_coil_support`

`material_coil_support` é a variável central de uma das hipóteses de negócio do projeto (Seção 4.2.1: alumínio vs. plástico no suporte da bobina). Inspecionando os valores reais, a coluna tem 3 categorias, não 2: `"Aluminium"`, `"Plastic"` e `"Aluminium Plate"` (linhas do Run 48). `"Aluminium Plate"` é uma variação de digitação do mesmo material `"Aluminium"` — mantê-la separada fragmentaria essa variável exatamente onde ela é mais importante para o projeto, corrompendo o teste da hipótese. A correção abaixo unifica as duas grafias.

In [ ]:
print("material_coil_support -- valores antes:", sorted(df['material_coil_support'].str.strip().unique()))

df['material_coil_support'] = df['material_coil_support'].str.strip().replace({
    'Aluminium Plate': 'Aluminium',
})

print("material_coil_support -- valores depois:", sorted(df['material_coil_support'].unique()))
assert df['material_coil_support'].nunique() == 2, "Esperado exatamente 2 materiais (Aluminium, Plastic)"

### 6.3 Normalização de `blower_direction`

Os valores reais do dataset misturam `"To back"`, `"To Front"`, `"To front"` e `"To Back"` — a mesma informação (sentido do fluxo de ar) escrita com capitalização inconsistente. A normalização usa `.replace()` sobre uma chave em minúsculas, e não `.map()` direto: `.map()` transformaria em `NaN` qualquer valor fora do dicionário, silenciosamente — e como esse `NaN` seria depois codificado pelo One-Hot como todas as colunas zeradas (sem gerar erro nem aparecer em uma checagem de nulos), essa falha passaria despercebida. O `assert` abaixo confirma **antes** de mapear que não existe nenhum valor fora do esperado, falhando ruidosamente em vez de corromper dados silenciosamente.

In [ ]:
mapa_direcao = {'to back': 'To Back', 'to front': 'To Front'}
chave_normalizada = df['blower_direction'].str.strip().str.lower()

nao_mapeados = chave_normalizada[~chave_normalizada.isin(mapa_direcao)].unique()
assert len(nao_mapeados) == 0, (
    f"blower_direction tem valor(es) não previstos na normalização: {nao_mapeados}. "
    "Atualize mapa_direcao antes de prosseguir -- não converter silenciosamente para NaN."
)

df['blower_direction'] = chave_normalizada.map(mapa_direcao)
print("blower_direction -- valores após normalização:", sorted(df['blower_direction'].unique()))

### 6.4 Outliers nas variáveis quantitativas

`temp` já foi analisada por IQR agrupado por `tc_name` na Seção 3: existem outliers, mas são plausíveis (refletem o `run_case` de cada teste) e foram mantidos. `active_limit` ainda não tinha sido verificada com evidência — uma versão anterior desta seção afirmava "sem outliers relevantes" sem nenhum cálculo, o que não é uma justificativa válida. O código abaixo corrige isso: como `active_limit` é um parâmetro de projeto que se repete para todas as linhas de um mesmo componente (não uma medição por linha), o IQR é calculado sobre os valores únicos por `tc_name`, evitando contar o mesmo limite dezenas de vezes.

In [ ]:
limites_por_componente = df.drop_duplicates(subset=['tc_name'])['active_limit']

q1 = limites_por_componente.quantile(0.25)
q3 = limites_por_componente.quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers_active_limit = limites_por_componente[
    (limites_por_componente < limite_inferior) | (limites_por_componente > limite_superior)
]

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Faixa esperada: [{limite_inferior}, {limite_superior}]")
print(f"Componentes com active_limit fora da faixa: {len(outliers_active_limit)} de {len(limites_por_componente)}")
outliers_active_limit

Assim como `temp`, nenhum valor de `active_limit` deve ser removido automaticamente caso apareça como outlier: limites de projeto diferentes refletem componentes diferentes por natureza, não erro de medição. **Esta célula, executada no Colab, é a evidência que sustenta (ou revisa) a conclusão sobre outliers em `active_limit`** — a conclusão passa a depender do resultado real, não de uma suposição.

### 6.5 Divisão treino/teste agrupada por `run_id`

Cada rodada (`run_id`) gera várias linhas — uma por termopar —, todas compartilhando a mesma configuração de teste. Uma divisão aleatória por linha deixaria leituras da mesma rodada tanto no treino quanto no teste, vazando informação de configuração entre os conjuntos (*data leakage*). Por isso a divisão é feita por grupo: todas as linhas de uma mesma `run_id` vão inteiramente para treino ou para teste. Ajustar o encoder e os scalers apenas no treino, já nesta etapa de preparação, evita que eles "vejam" o conjunto de teste durante o ajuste.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df[CATEGORICAS + NUMERICAS].copy()
y = df[TARGET].copy()
grupos = df['run_id']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_treino, idx_teste = next(splitter.split(X, y, groups=grupos))

X_train, X_test = X.iloc[idx_treino], X.iloc[idx_teste]
y_train, y_test = y.iloc[idx_treino], y.iloc[idx_teste]

runs_treino = set(df.loc[idx_treino, 'run_id'])
runs_teste = set(df.loc[idx_teste, 'run_id'])

print(f"Linhas treino: {len(X_train)} | Linhas teste: {len(X_test)}")
print(f"Rodadas em ambos os conjuntos (deve ser 0): {len(runs_treino & runs_teste)}")
assert len(runs_treino & runs_teste) == 0, "Vazamento: alguma run_id aparece em treino e teste"

### 6.6 Codificação das variáveis categóricas (fit apenas no treino)

Nenhuma variável foi codificada com Label Encoding, pois todas são nominais — não há ordem entre, por exemplo, `run_case = "Pyro"` e `"Broil C2"`. A técnica usada é **One-Hot Encoding**, com `handle_unknown='ignore'` (uma categoria nova, vista só no teste, vira todas as colunas zeradas em vez de gerar erro) e `drop='if_binary'` (`deflector_hole`, a única variável com 2 categorias, vira 1 coluna 0/1 em vez de 2 colunas redundantes).

`drop='if_binary'` só remove uma categoria em variáveis binárias — diferente de um `drop='first'` geral, que removeria uma categoria de *todas* as colunas (necessário apenas se o modelo final for de regressão linear, para evitar a *dummy variable trap*/multicolinearidade perfeita entre as dummies e o intercepto). Como a Seção 4.3 (modelagem) ainda não define o algoritmo, mantemos todas as categorias das variáveis não binárias: isso não prejudica modelos baseados em árvore (o caso mais provável dado o tipo de problema) e, se um modelo linear for escolhido depois, basta remover uma coluna por variável antes do ajuste.

O encoder é ajustado (`fit`) somente com `X_train`: se fosse ajustado com o dataset completo antes da divisão, o vocabulário de categorias "vazaria" informação do teste para o treino.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False)
encoder.fit(X_train[CATEGORICAS])

colunas_codificadas = encoder.get_feature_names_out(CATEGORICAS)

X_train_cat = pd.DataFrame(
    encoder.transform(X_train[CATEGORICAS]), columns=colunas_codificadas, index=X_train.index,
)
X_test_cat = pd.DataFrame(
    encoder.transform(X_test[CATEGORICAS]), columns=colunas_codificadas, index=X_test.index,
)

# Confere que o número de colunas geradas bate com o esperado (soma de categorias-1
# para deflector_hole, categorias completas para as demais) -- pega, por exemplo,
# uma categoria que virou 2 dummies por causa de espaço em branco não tratado.
categorias_por_coluna = {col: X_train[col].nunique() for col in CATEGORICAS}
esperado = sum(
    (n - 1 if col == 'deflector_hole' else n) for col, n in categorias_por_coluna.items()
)
print(f"Categorias por coluna (no treino): {categorias_por_coluna}")
print(f"Colunas geradas: {len(colunas_codificadas)} | Esperado: {esperado}")
assert len(colunas_codificadas) == esperado, "Número de colunas do One-Hot não bate com o esperado"

X_train_cat.head()

### 6.7 Normalização das variáveis quantitativas (fit apenas no treino)

`active_limit` representa um conjunto fechado de limites de projeto (poucas dezenas de valores possíveis), e a Seção 6.4 confirmou, com evidência, a ausência de outliers relevantes — por isso foi normalizada com **MinMaxScaler**, que leva os valores para `[0, 1]` preservando a posição relativa de cada limite.

`temp`, a variável-alvo, tem outliers legítimos confirmados na Seção 3 (`run_case` Pyro ~50 °C vs. Broil C2 ~220 °C). Caso o modelo escolhido na Seção 4.3 seja sensível à escala do alvo (regressão linear, KNN, rede neural), a técnica recomendada é o **RobustScaler**, não o StandardScaler: ele centraliza pela mediana e escala pelo intervalo interquartil, valores menos sensíveis a esses extremos do que média/desvio padrão. **Interpretação prática:** diferente do MinMaxScaler, o resultado não fica limitado a `[0, 1]` — um valor de `temp` escalonado igual a `2,0`, por exemplo, significa "2 IQRs acima da mediana do treino", e pode ultrapassar livremente esse intervalo em leituras muito quentes (ex.: Oven RTD Connector Body, ~200–270 °C). Modelos baseados em árvore (Random Forest, Gradient Boosting) dispensam esse escalonamento inteiramente, tanto em `X` quanto em `y`.

Tanto o `scaler_active_limit` quanto o `scaler_temp` são ajustados (`fit`) apenas com os dados de treino; o teste passa somente por `transform`, usando os parâmetros aprendidos no treino — a mesma disciplina aplicada ao encoder da Seção 6.6.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

scaler_active_limit = MinMaxScaler()
X_train_num = pd.DataFrame(
    scaler_active_limit.fit_transform(X_train[NUMERICAS]), columns=NUMERICAS, index=X_train.index,
)
X_test_num = pd.DataFrame(
    scaler_active_limit.transform(X_test[NUMERICAS]), columns=NUMERICAS, index=X_test.index,
)

# Escalonamento do alvo é opcional -- útil só se o modelo escolhido na Seção 4.3/4.4
# for sensível à escala (regressão linear, KNN, rede neural). Modelos baseados em
# árvore devem usar y_train/y_test originais, em °C, sem esta transformação.
scaler_temp = RobustScaler()
y_train_scaled = pd.Series(
    scaler_temp.fit_transform(y_train.to_frame()).ravel(), index=y_train.index, name='temp_scaled',
)
y_test_scaled = pd.Series(
    scaler_temp.transform(y_test.to_frame()).ravel(), index=y_test.index, name='temp_scaled',
)

print(f"active_limit (treino) -- min: {X_train_num['active_limit'].min():.3f}, max: {X_train_num['active_limit'].max():.3f}")
print(f"temp_scaled (treino) -- mediana: {y_train_scaled.median():.3f}, IQR: {(y_train_scaled.quantile(0.75) - y_train_scaled.quantile(0.25)):.3f}")

### 6.8 Construção do dataset final pré-processado

O dataset final (`X_train_final`/`X_test_final`) contém **apenas** as features prontas para modelagem — sem identificadores nem texto livre. Quem for usar este dataset na Seção 4.3 não precisa lembrar de excluir nada de novo.

In [ ]:
X_train_final = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_final = pd.concat([X_test_num, X_test_cat], axis=1)

print(f"X_train_final: {X_train_final.shape}")
print(f"X_test_final: {X_test_final.shape}")
X_train_final.head()

### 6.9 Validação das transformações

In [ ]:
import numpy as np

# Linhas preservadas
assert len(X_train_final) + len(X_test_final) == len(X)
print("Linhas preservadas:", len(X_train_final) + len(X_test_final), "==", len(X))

# Nenhum NaN introduzido, checado por coluna (não só a soma agregada)
nulos_train = X_train_final.isnull().sum()
nulos_teste = X_test_final.isnull().sum()
assert nulos_train.sum() == 0, f"NaN introduzido no treino:\n{nulos_train[nulos_train > 0]}"
assert nulos_teste.sum() == 0, f"NaN introduzido no teste:\n{nulos_teste[nulos_teste > 0]}"
print("NaN no treino:", nulos_train.sum(), "| NaN no teste:", nulos_teste.sum())

# Nenhum valor infinito
assert not np.isinf(X_train_final.to_numpy(dtype=float)).any()
assert not np.isinf(X_test_final.to_numpy(dtype=float)).any()

# Colunas do One-Hot contêm apenas 0/1
valores_onehot = X_train_final[colunas_codificadas].to_numpy()
assert set(np.unique(valores_onehot)).issubset({0.0, 1.0})
print("Valores únicos nas colunas codificadas:", np.unique(valores_onehot))

# active_limit escalonado dentro de [0, 1] no treino (MinMaxScaler)
assert 0.0 <= X_train_final['active_limit'].min() and X_train_final['active_limit'].max() <= 1.0
print(f"active_limit (treino) dentro de [0, 1]: min={X_train_final['active_limit'].min():.3f}, max={X_train_final['active_limit'].max():.3f}")

# temp_scaled com mediana ~0 no treino (RobustScaler)
print(f"temp_scaled (treino) -- mediana: {y_train_scaled.median():.3f} (esperado próximo de 0)")

### 6.10 Comparação antes × depois

O aumento no número de colunas abaixo é esperado e não representa perda de dados: o One-Hot Encoding expande cada variável nominal em uma coluna por categoria (ou uma única coluna para a variável binária).

In [ ]:
comparacao = pd.DataFrame({
    'Aspecto': [
        'Linhas (treino+teste)', 'Colunas em X', 'Variáveis categóricas não codificadas',
        'Variáveis quantitativas normalizadas', 'Valores ausentes em X',
    ],
    'Antes': [len(X), X.shape[1], len(CATEGORICAS), 0, int(X.isnull().sum().sum())],
    'Depois': [
        len(X_train_final) + len(X_test_final), X_train_final.shape[1], 0, len(NUMERICAS),
        int(X_train_final.isnull().sum().sum() + X_test_final.isnull().sum().sum()),
    ],
})
comparacao

### 6.11 Visualização antes × depois

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(X_train['active_limit'], bins=15, color='steelblue', edgecolor='white')
axes[0, 0].set_title('active_limit -- antes (°C)')

axes[0, 1].hist(X_train_final['active_limit'], bins=15, color='darkorange', edgecolor='white')
axes[0, 1].set_title('active_limit -- depois (MinMaxScaler, [0,1])')

axes[1, 0].hist(y_train, bins=40, color='steelblue', edgecolor='white')
axes[1, 0].set_title('temp -- antes (°C)')

axes[1, 1].hist(y_train_scaled, bins=40, color='darkorange', edgecolor='white')
axes[1, 1].set_title('temp -- depois (RobustScaler)')

plt.tight_layout()
plt.show()

### 6.12 Decisões de julgamento registradas

Duas decisões desta seção não são puramente mecânicas e precisam ficar documentadas, para que outra pessoa do grupo (ou o professor) não as confunda com descuido:

- **`run_case` mantém `"Broil C2"`/`"Broil Case 2"` e `"Broil Qualitative"`/`"Broil Qual"`/`"Qualitative"` como categorias separadas.** Essas grafias foram investigadas e confirmadas pelo grupo como configurações de teste efetivamente diferentes, não a mesma categoria escrita de formas distintas — por isso não foram unificadas antes do One-Hot Encoding.
- **`material_coil_support` teve `"Aluminium Plate"` corrigido para `"Aluminium"`** (Seção 6.2), pois essa variação de digitação fragmentava artificialmente a variável central de uma das hipóteses de negócio do projeto (alumínio vs. plástico).